# 핏메이트 — 카탈로그 CLIP 임베딩 생성 (Colab GPU, 1회성)

카탈로그 상품 이미지들을 CLIP 벡터로 변환해 `clothes_embeddings.json`을 만든다.
이 파일은 Node.js 서버(`src/server/routes/similar.ts`)가 읽어서 순수 JS 코사인
유사도로 '유사 상품 추천'을 계산하는 데 쓰인다 — 런타임에는 GPU/외부 API 호출이
전혀 필요 없다.

카탈로그가 몇 장뿐일 때는 `npm run embeddings`(로컬 CPU, `@xenova/transformers`)로도
충분하다. 이 노트북은 상품 수가 많아져(수백~수천 장) 로컬 CPU로는 느릴 때 Colab의
무료 GPU로 한 번에 처리하기 위한 용도다.

**사용법**
1. Runtime > Change runtime type > GPU 선택
2. 아래 셀을 순서대로 실행
3. 상품 이미지들을 `images/` 폴더에 업로드 (파일명 = 상품 id, 예: `top-001.jpg`)
4. 마지막 셀 실행 후 생성된 `clothes_embeddings.json`을 다운받아 `assets/`에 덮어쓰기

In [ ]:
!pip install -q open_clip_torch pillow

In [ ]:
import os

IMAGES_DIR = 'images'  # 상품 이미지 폴더, 파일명이 곧 상품 id
os.makedirs(IMAGES_DIR, exist_ok=True)
print(f'{IMAGES_DIR}/ 에 상품 이미지를 업로드한 뒤 다음 셀로 진행하세요.')
print('예: images/top-001.jpg, images/bottom-014.png ...')

In [ ]:
# Colab 좌측 파일 탐색기로 드래그 앤 드롭 업로드하거나, 아래로 직접 업로드 위젯 사용
from google.colab import files
uploaded = files.upload()
for name, content in uploaded.items():
    with open(os.path.join(IMAGES_DIR, name), 'wb') as f:
        f.write(content)
print(f'{len(uploaded)}개 파일 업로드 완료')

In [ ]:
import json
import torch
import open_clip
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# @xenova/transformers 쪽에서 쓰는 Xenova/clip-vit-base-patch32와 동일 계열
# (openai ViT-B/32) — 두 파이프라인의 벡터가 같은 공간에 있어야 서로 호환된다.
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
model.eval().to(device)

In [ ]:
embeddings = {}

image_files = sorted(
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
)

for filename in image_files:
    item_id = os.path.splitext(filename)[0]
    image = preprocess(Image.open(os.path.join(IMAGES_DIR, filename)).convert('RGB'))
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encode_image(image)
        features = features / features.norm(dim=-1, keepdim=True)  # L2 정규화

    embeddings[item_id] = features.squeeze(0).cpu().tolist()
    print(f'{item_id} <- {filename} ({len(embeddings[item_id])}차원)')

with open('clothes_embeddings.json', 'w') as f:
    json.dump(embeddings, f)

print(f'\n총 {len(embeddings)}개 임베딩 -> clothes_embeddings.json')

In [ ]:
from google.colab import files
files.download('clothes_embeddings.json')
# 다운받은 파일을 프로젝트의 assets/clothes_embeddings.json 에 덮어쓰면 된다.